# Bortle layers for laser cutting: whole world

World version of `laser_layers.ipynb`. It downloads the planet from lightpollutionmap.app (cached in `lp_world.png`) and writes one cut file per Bortle level.

* `laser_layers_world/bortle_1.svg` … `bortle_9.svg` are red hairline outlines, sized in mm.
* `laser_layers_world/bortle_layers.pdf` holds the same outlines, one page per level.

**How the layers stack.** Sheet *k* covers every land area with Bortle ≥ *k*. Put sheet 1 at the bottom and sheet 9 on top. Every sheet gets the same rectangular frame so they line up.

**Projection.** Web Mercator, like the website. The map is cropped at `LAT_MIN` and `LAT_MAX` because Mercator stretches without limit near the poles.

**Pipeline.** Settings → download tiles → classify colours → land mask and crop → trace outlines → write files → preview.

In [ ]:
# Cell 1: settings and map projection helpers

import io
import json
import os
import time
from math import floor

import numpy as np
import requests
from PIL import Image, ImageDraw, ImageFilter
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# ---------------------------------------------------------------- files
MOSAIC_PATH = "lp_world.png"            # cached download, delete to fetch again
LAND_GEOJSON_PATH = "../data/ne_50m_land.geojson"
OUT_DIR = "laser_layers_world"

# ---------------------------------------------------------------- map extent
ZOOM = 3                  # 2**3 = 8 x 8 tiles for the whole world
TILE_SIZE = 512           # source tiles are 1024 px, 512 keeps memory reasonable
LON_MIN, LON_MAX = -180.0, 180.0
LAT_MIN, LAT_MAX = -58.0, 78.0    # cuts off Antarctica and the far Arctic

# ---------------------------------------------------------------- cut settings
SHEET_WIDTH_MM = 600      # height follows from the map
SMOOTH_PX = 2             # blur before tracing, bigger = smoother outlines
MIN_AREA_MM2 = 3          # drop shapes too small to cut
FRAME = True              # same outer rectangle on every sheet, for alignment
# Stroke colour. Most laser software treats pure red as "cut".
CUT_COLOR = "#FF0000"
STROKE_MM = 0.01          # hairline

# ---------------------------------------------------------------- tile server
TILE_URL = ("https://api.lightpollutionmap.app/api/lightpollution/image-tiles/"
            "2025/{z}/{x}/{y}.png?v=v6-20260724")
HEADERS = {"User-Agent": "Mozilla/5.0", "Referer": "https://lightpollutionmap.app/"}
REQUEST_DELAY_S = 0.05    # pause between tile requests


def lon_to_px(lon):
    """Convert longitude to an x pixel coordinate in the full world image."""
    world_px = 2 ** ZOOM * TILE_SIZE
    return (np.asarray(lon) + 180) / 360 * world_px


def lat_to_px(lat):
    """Convert latitude to a y pixel coordinate (Web Mercator)."""
    world_px = 2 ** ZOOM * TILE_SIZE
    rad = np.radians(np.clip(lat, -85, 85))
    mercator_y = np.log(np.tan(rad) + 1 / np.cos(rad))
    return (1 - mercator_y / np.pi) / 2 * world_px


# Top-left corner of the mosaic in world pixels, snapped to a tile edge.
MOSAIC_X0 = floor(lon_to_px(LON_MIN) / TILE_SIZE) * TILE_SIZE
MOSAIC_Y0 = floor(lat_to_px(LAT_MAX) / TILE_SIZE) * TILE_SIZE

## Download the world mosaic
Skipped when `lp_world.png` already exists.

In [ ]:
# Cell 2: download the world mosaic

def download_world_mosaic(path):
    """Download every tile covering the map extent and stitch them into one PNG."""
    tile_cols = range(int(MOSAIC_X0 // TILE_SIZE),
                      int(np.ceil(lon_to_px(LON_MAX) / TILE_SIZE)))
    tile_rows = range(int(MOSAIC_Y0 // TILE_SIZE),
                      int(lat_to_px(LAT_MIN) // TILE_SIZE) + 1)

    mosaic = Image.new("RGB", (len(tile_cols) * TILE_SIZE, len(tile_rows) * TILE_SIZE))
    for row, tile_y in enumerate(tile_rows):
        for col, tile_x in enumerate(tile_cols):
            url = TILE_URL.format(z=ZOOM, x=tile_x, y=tile_y)
            response = requests.get(url, headers=HEADERS, timeout=30)
            if response.status_code != 200:
                print(f"tile ({tile_x}, {tile_y}) failed: HTTP {response.status_code}")
                continue
            tile = Image.open(io.BytesIO(response.content)).convert("RGB")
            tile = tile.resize((TILE_SIZE, TILE_SIZE), Image.NEAREST)
            mosaic.paste(tile, (col * TILE_SIZE, row * TILE_SIZE))
            time.sleep(REQUEST_DELAY_S)
        print(f"tile row {tile_y} done")
    mosaic.save(path)


if not os.path.exists(MOSAIC_PATH):
    download_world_mosaic(MOSAIC_PATH)
print("mosaic size:", Image.open(MOSAIC_PATH).size)

## Classify colours into Bortle levels
Each pixel gets the Bortle level of the nearest palette colour. The site uses 14 colours, mapped onto Bortle 1 to 9.

In [ ]:
# Cell 3: classify colours into Bortle levels

# The 14 colours of the lightpollutionmap.app 2025 tiles, darkest to brightest.
PALETTE = np.array([
    (0, 0, 0), (34, 34, 34), (66, 66, 66),      # black / greys
    (20, 47, 114), (33, 84, 216),               # blues
    (15, 87, 20), (31, 161, 42),                # greens
    (110, 100, 30), (184, 166, 37),             # yellows
    (191, 100, 30), (253, 150, 80),             # oranges
    (240, 84, 72), (250, 155, 140), (255, 255, 255),  # reds / white
], dtype=np.int32)

# Bortle level for each colour above. Approximate, adjust to taste.
PALETTE_TO_BORTLE = np.array([1, 1, 2, 3, 4, 4, 5, 5, 6, 6, 7, 8, 9, 9])


def classify_bortle(rgb_image):
    """Map every pixel to the Bortle level of its nearest palette colour."""
    pixels = rgb_image.reshape(-1, 3).astype(np.int32)
    best_dist = np.full(len(pixels), np.inf, dtype=np.float32)
    best_class = np.zeros(len(pixels), dtype=np.int8)

    for index, colour in enumerate(PALETTE):
        dist = ((pixels - colour) ** 2).sum(axis=1)   # squared RGB distance
        closer = dist < best_dist
        best_dist[closer] = dist[closer]
        best_class[closer] = index

    return PALETTE_TO_BORTLE[best_class].reshape(rgb_image.shape[:2])


mosaic_rgb = np.array(Image.open(MOSAIC_PATH).convert("RGB"))
bortle = classify_bortle(mosaic_rgb)
print("Bortle levels present:", np.unique(bortle))

## Land mask and crop
Sea pixels are removed with the Natural Earth land polygons, then the map is cropped to the lon/lat box.

In [ ]:
# Cell 4: land mask + crop to the region

def rasterize_land(geojson_path, width, height):
    """Draw the land polygons into a boolean mask aligned with the mosaic."""
    land_img = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(land_img)

    with open(geojson_path) as f:
        features = json.load(f)["features"]

    for feature in features:
        geometry = feature["geometry"]
        if geometry["type"] == "MultiPolygon":
            polygons = geometry["coordinates"]
        else:
            polygons = [geometry["coordinates"]]

        for polygon in polygons:
            for ring_index, ring in enumerate(polygon):
                ring = np.asarray(ring)
                xs = lon_to_px(ring[:, 0]) - MOSAIC_X0
                ys = lat_to_px(ring[:, 1]) - MOSAIC_Y0
                fill = 255 if ring_index == 0 else 0
                draw.polygon(list(zip(xs, ys)), fill=fill)

    return np.array(land_img) > 127


land = rasterize_land(LAND_GEOJSON_PATH, bortle.shape[1], bortle.shape[0])

# Crop both maps to the requested lon/lat box.
col0 = int(lon_to_px(LON_MIN) - MOSAIC_X0)
col1 = int(lon_to_px(LON_MAX) - MOSAIC_X0)
row0 = int(lat_to_px(LAT_MAX) - MOSAIC_Y0)
row1 = int(lat_to_px(LAT_MIN) - MOSAIC_Y0)
bortle = bortle[row0:row1, col0:col1]
land = land[row0:row1, col0:col1]

MAP_HEIGHT_PX, MAP_WIDTH_PX = bortle.shape
MM_PER_PX = SHEET_WIDTH_MM / MAP_WIDTH_PX
SHEET_HEIGHT_MM = MAP_HEIGHT_PX * MM_PER_PX

print(f"crop {MAP_WIDTH_PX} x {MAP_HEIGHT_PX} px  ->  "
      f"sheet {SHEET_WIDTH_MM:.0f} x {SHEET_HEIGHT_MM:.0f} mm")

## Trace outlines per Bortle level
The mask is blurred and contoured at half height, which gives smooth outlines. Tiny shapes are dropped.

In [ ]:
# Cell 5: trace outlines per Bortle level

def ring_area(points):
    """Area of a closed ring (shoelace formula)."""
    x, y = points[:, 0], points[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))


def trace_outlines(mask):
    """Smooth outlines of a boolean mask, in sheet millimetres."""
    # Pad wider than the blur, so shapes touching the edge still close.
    pad = 3 * SMOOTH_PX + 2
    padded = np.pad(mask, pad).astype(np.uint8) * 255
    blurred = Image.fromarray(padded).filter(ImageFilter.GaussianBlur(SMOOTH_PX))
    soft = np.array(blurred, dtype=float)

    contour = plt.contour(soft, levels=[127.5])
    plt.close()

    rings = []
    for ring in contour.get_paths()[0].to_polygons(closed_only=False):
        ring = (ring - pad) * MM_PER_PX                          # px -> mm
        ring = np.clip(ring, 0, [SHEET_WIDTH_MM, SHEET_HEIGHT_MM])  # follow sheet border
        if ring_area(ring) >= MIN_AREA_MM2:
            rings.append(ring)
    return rings


# Cumulative layers: sheet k covers all land with Bortle >= k.
layers = {}
for level in range(1, 10):
    layers[level] = trace_outlines(land & (bortle >= level))
    print(f"Bortle {level}: {len(layers[level])} outlines")

## Write cut files
One SVG per level and one multi-page PDF.

In [ ]:
# Cell 6: write SVGs (one per level) and one multi-page PDF

def ring_to_svg_path(ring):
    """Format a ring as an SVG path string."""
    return "M" + " L".join(f"{x:.3f},{y:.3f}" for x, y in ring) + " Z"


def frame_svg_path(width, height):
    """SVG path of the alignment rectangle around the sheet."""
    return f"M0,0 L{width:.3f},0 L{width:.3f},{height:.3f} L0,{height:.3f} Z"


def write_svg(path, level, rings, width, height):
    """Write one Bortle level as an SVG cut file sized in mm."""
    paths = [ring_to_svg_path(r) for r in rings]
    if FRAME:
        paths.append(frame_svg_path(width, height))
    body = "\n".join(f'  <path d="{d}"/>' for d in paths)

    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" '
        f'width="{width:.3f}mm" height="{height:.3f}mm" '
        f'viewBox="0 0 {width:.3f} {height:.3f}">\n'
        f'<g id="bortle_{level}" fill="none" stroke="{CUT_COLOR}" '
        f'stroke-width="{STROKE_MM}">\n{body}\n</g>\n</svg>\n'
    )
    with open(path, "w") as f:
        f.write(svg)


def write_pdf(path, layers, width, height):
    """Write all levels into one PDF, one page per level, at true size."""
    inch_per_mm = 1 / 25.4
    with PdfPages(path) as pdf:
        for rings in layers.values():
            fig = plt.figure(figsize=(width * inch_per_mm, height * inch_per_mm))
            ax = fig.add_axes([0, 0, 1, 1])
            for ring in rings:
                closed = np.vstack([ring, ring[:1]])
                ax.plot(closed[:, 0], closed[:, 1], color=CUT_COLOR, lw=0.1)
            if FRAME:
                ax.plot([0, width, width, 0, 0], [0, 0, height, height, 0],
                        color=CUT_COLOR, lw=0.1)
            ax.set_xlim(0, width)
            ax.set_ylim(height, 0)      # y grows downward, like the SVG
            ax.axis("off")
            pdf.savefig(fig)
            plt.close(fig)


os.makedirs(OUT_DIR, exist_ok=True)
for level, rings in layers.items():
    write_svg(f"{OUT_DIR}/bortle_{level}.svg", level, rings, SHEET_WIDTH_MM, SHEET_HEIGHT_MM)
write_pdf(f"{OUT_DIR}/bortle_layers.pdf", layers, SHEET_WIDTH_MM, SHEET_HEIGHT_MM)
print("written to", OUT_DIR)

## Preview all layers

In [ ]:
# Cell 7: preview all layers

def plot_preview(layers):
    """Show the nine layers in a 3 x 3 grid."""
    fig, axes = plt.subplots(3, 3, figsize=(15, 15 * MAP_HEIGHT_PX / MAP_WIDTH_PX))
    for (level, rings), ax in zip(layers.items(), axes.ravel()):
        for ring in rings:
            closed = np.vstack([ring, ring[:1]])
            ax.plot(closed[:, 0], closed[:, 1], color="red", lw=0.4)
        ax.set_xlim(0, SHEET_WIDTH_MM)
        ax.set_ylim(SHEET_HEIGHT_MM, 0)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f"Bortle {level}+  ({len(rings)} outlines)")
    plt.tight_layout()
    plt.show()


plot_preview(layers)